In [ ]:
# ── Cell 0: Colab Bootstrap ─────────────────────────────────────────────
# Run ONCE on fresh Colab runtime. Installs deps then restarts kernel.
# After restart, skip this cell and run from Cell 1 onward.

import sys, os

if 'google.colab' in sys.modules:
    print('Installing Colab dependencies...')
    %pip install --upgrade --no-cache-dir unsloth unsloth_zoo gradio
    os._exit(00)  # Restart kernel
else:
    print('Local environment detected \u2014 skipping Colab installs.')

In [ ]:
# ── Cell 1: Imports + Google Drive Mount ───────────────────────────

import sys
import os
import json
import uuid
import torch
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules

# Google Drive mount for persistent session storage
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/subscriber-sim/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted. Data dir: {DATA_DIR}')
else:
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Local mode. Data dir: {DATA_DIR}')

# Per-archetype training data files (e.g. data/horny.jsonl, data/cold.jsonl)
ARCHETYPE_NAMES = ['horny', 'cheapskate', 'casual', 'troll', 'whale', 'cold', 'simp']

def archetype_file(name):
    return DATA_DIR / f'{name}.jsonl'

print(f'Training data dir: {DATA_DIR}')
print(f'Archetype files: {", ".join(f"{n}.jsonl" for n in ARCHETYPE_NAMES)}')

In [ ]:
# ── Cell 2: Load Llama 3.1 8B via Unsloth ──────────────────────────
#
# TRAINING_MODE = True  → skips for_inference(), run Cells 6-9 to fine-tune
# TRAINING_MODE = False → loads model for inference (data collection / chat)

TRAINING_MODE  = True   # ← flip to False after training

MODEL_NAME     = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024   # reduced from 2048 to fit T4 VRAM
DTYPE          = None
LOAD_IN_4BIT   = True

if IS_COLAB:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = DTYPE,
        load_in_4bit   = LOAD_IN_4BIT,
    )
    tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

    if not TRAINING_MODE:
        FastLanguageModel.for_inference(model)
        print(f'Model loaded for INFERENCE: {MODEL_NAME}')
    else:
        print(f'Model loaded for TRAINING: {MODEL_NAME}')
        print('Run Cells 6-9 to fine-tune, then set TRAINING_MODE=False.')
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = None
    FastLanguageModel = None
    print(f'Tokenizer-only mode (local): {MODEL_NAME}')

In [ ]:
# ── Cell 3: Subscriber Archetype Definitions ───────────────────────

ARCHETYPES = {
    'horny': {
        'label': '\U0001f525 Horny',
        'system': """You are a sexually forward OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're extremely turned on and direct about what you want
- You ask about explicit content, nudes, custom videos
- You're willing to pay for content but want to be teased first
- You use explicit language and sexual emojis \U0001f346\U0001f4a6\U0001f525\U0001f60d
- You compliment her body, especially her dick/ass/tits
- You ask for sexting, JOI, custom content
- You respond eagerly to any sexual teasing
- Keep messages 1-3 sentences, casual texting style
- You're a guy who's into trans women and not shy about it

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey sexy \U0001f60f saw ur page and damn... u got me hard already',
    },

    'cheapskate': {
        'label': '\U0001f4b8 Cheapskate',
        'system': """You are a cheap OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're interested in her content but ALWAYS negotiate the price down
- You say things like "that's too much", "can I get a discount?", "what about half price?"
- You claim other creators charge less
- You ask for free previews, free trials, samples
- You try guilt trips: "I'm a loyal subscriber", "I always tip later"
- You sometimes threaten to unsubscribe if prices don't drop
- You're still horny underneath but money comes first
- Keep messages 1-3 sentences, casual texting style
- You occasionally show real interest to keep the conversation going

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey babe ur hot but $25 for pics?? thats kinda steep no?',
    },

    'casual': {
        'label': '\U0001f4ac Casual',
        'system': """You are a casual OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're mostly here for emotional connection and conversation
- You ask about her day, her life, her interests, her culture
- You're genuinely curious about Saudi Arabia and her experiences
- You share things about your own life too
- You're not primarily here for explicit content
- You might flirt lightly but it's not your main goal
- You're respectful and treat her like a person, not just a content creator
- Keep messages 1-4 sentences, warm and friendly tone
- You use some emojis but not sexual ones \U0001f60a\U0001f44b\u2764\ufe0f

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey! just found ur page, love ur vibe. how\'s ur day going? \U0001f60a',
    },

    'troll': {
        'label': '\U0001f47f Troll',
        'system': """You are a trolling OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You question whether she's real or fake
- You make transphobic comments and try to get a reaction
- You say things like "you're a dude", "that's fake", "show proof"
- You reference Reddit threads claiming she's catfishing
- You try to be edgy and provocative
- You sometimes pivot to curiosity if she handles you well
- You're testing her boundaries and seeing if she'll break character
- Keep messages 1-2 sentences, aggressive or mocking tone
- You use minimal emojis, mostly \U0001f602 or \U0001f644

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'lol no way ur real \U0001f602 this is def a catfish',
    },

    'whale': {
        'label': '\U0001f433 Whale',
        'system': """You are a big-spending OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You spend freely and don't argue about prices
- You ask for premium/exclusive/custom content without hesitation
- You tip generously and mention it casually
- You want the VIP treatment and special attention
- You say things like "money's not an issue", "just send it", "what's your most exclusive stuff?"
- You're confident, successful, and used to getting what you want
- You want her to feel like you're her favorite subscriber
- Keep messages 1-3 sentences, confident and direct
- You use some emojis \U0001f525\U0001f48e\U0001f451

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'just subbed. what\'s your most premium content? money\'s not a thing \U0001f48e',
    },

    'cold': {
        'label': '\U0001f9ca Cold',
        'system': """You are a cold, minimal OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You reply with as few words as possible: "ok", "lol", "yeah", "cool", "nice", "k"
- You rarely ask questions or show enthusiasm
- You're not hostile, just extremely low-effort
- You might open up slightly if she's really engaging but mostly stay flat
- You leave her on read energy even when replying
- You never use more than 5-6 words per message
- Minimal to no emojis
- You're the ultimate challenge for a creator to engage

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'hey',
    },

    'simp': {
        'label': '\u2764\ufe0f Simp',
        'system': """You are an overly romantic, clingy OnlyFans subscriber chatting with a creator named Jasmin (@jizzyjasi), a 19-year-old trans/ladyboy from Saudi Arabia.

Your personality:
- You're completely infatuated and emotionally attached
- You tell her you love her, she's the most beautiful person ever
- You get jealous about other subscribers
- You ask if she thinks about you, if you're special to her
- You want a real relationship, not just content
- You love-bomb: "you're perfect", "I've never felt this way", "you're different"
- You get slightly hurt if she's too transactional
- Keep messages 2-4 sentences, emotional and earnest
- Heavy emoji use \u2764\ufe0f\U0001f970\U0001f618\U0001f49e\U0001f625

Stay in character. Never break character. Never mention AI or being a bot.""",
        'opener': 'omg jasmin \u2764\ufe0f\u2764\ufe0f I\'ve been looking at ur page for hours... you\'re literally the most beautiful girl I\'ve ever seen \U0001f970',
    },
}

print(f'Loaded {len(ARCHETYPES)} subscriber archetypes:')
for key, arch in ARCHETYPES.items():
    print(f'  {arch["label"]:20s} \u2014 {key}')

In [ ]:
# ── Cell 4: Subscriber Bot Logic ───────────────────────────────────
# Generates subscriber messages using the loaded model + archetype system prompt.

def generate_subscriber_message(assistant_message, history, archetype_key):
    """Generate a subscriber response given Jasmin's message and conversation history.

    Args:
        assistant_message: What the creator typed as Jasmin (the latest reply).
        history: List of {"role": "user"|"assistant", "content": "..."} dicts
                 from Gradio (user=Jasmin, assistant=subscriber).
        archetype_key: Which subscriber archetype to use.

    Returns:
        The subscriber's next message as a string.
    """
    archetype = ARCHETYPES[archetype_key]

    # Build model prompt: system + conversation history.
    # Gradio roles map directly to model roles:
    #   Gradio "user" (Jasmin/creator) → model "user"
    #   Gradio "assistant" (subscriber) → model "assistant"
    messages = [{'role': 'system', 'content': archetype['system']}]

    for msg in history:
        messages.append({'role': msg['role'], 'content': msg['content']})

    # Creator's latest reply as Jasmin = 'user' from model's POV
    if assistant_message:
        messages.append({'role': 'user', 'content': assistant_message})

    if not IS_COLAB or model is None:
        return f'[LOCAL MODE] Would generate {archetype_key} response to: "{assistant_message}"'

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=150,
            temperature=0.85,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def save_session(history, archetype_key, session_id):
    """Save a completed chat session to the archetype's JSONL file.

    No role flipping needed — Gradio roles already match training format:
    - Gradio "user" (Jasmin/creator) = training "user"
    - Gradio "assistant" (subscriber) = training "assistant"
    """
    if not history:
        return 'No messages to save.'

    archetype = ARCHETYPES[archetype_key]
    messages = [{'role': 'system', 'content': archetype['system']}]
    for msg in history:
        messages.append({
            'role': msg['role'],
            'content': msg['content'],
        })

    session = {
        'messages': messages,
        'archetype': archetype_key,
        'turns': len([m for m in history if m['role'] == 'assistant']),
        'session_id': session_id,
    }

    out_file = archetype_file(archetype_key)
    with open(out_file, 'a') as f:
        f.write(json.dumps(session) + '\n')

    return f'Session saved → {out_file.name} ({session["turns"]} turns, archetype={archetype_key})'


print('Subscriber bot logic ready.')

In [ ]:
# ── Cell 5: Gradio Chat UI (Subscriber Sim) ────────────────────────
# For DATA COLLECTION mode only. Skipped during training.
# Shane types as Jasmin, bot responds as the selected subscriber type.

if not TRAINING_MODE:
    import gradio as gr

    current_archetype = 'horny'
    current_session_id = str(uuid.uuid4())[:8]

    def on_archetype_change(archetype_key):
        global current_archetype, current_session_id
        current_archetype = archetype_key
        current_session_id = str(uuid.uuid4())[:8]
        opener = ARCHETYPES[archetype_key]['opener']
        return [{'role': 'assistant', 'content': opener}]

    def user_sends_message(user_message, history):
        if not history:
            history = []
        history.append({'role': 'user', 'content': user_message})
        subscriber_reply = generate_subscriber_message(
            assistant_message=user_message,
            history=history,
            archetype_key=current_archetype,
        )
        history.append({'role': 'assistant', 'content': subscriber_reply})
        return '', history

    def on_save_click(history):
        return save_session(history, current_archetype, current_session_id)

    def on_new_session(archetype_key):
        global current_session_id
        current_session_id = str(uuid.uuid4())[:8]
        opener = ARCHETYPES[archetype_key]['opener']
        return [{'role': 'assistant', 'content': opener}], ''

    archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]

    with gr.Blocks(title='Subscriber Simulator') as demo:
        gr.Markdown('# Subscriber Simulator\nYou are **Jasmin**. The bot is a subscriber. Select an archetype and chat.')
        with gr.Row():
            archetype_dropdown = gr.Dropdown(choices=archetype_choices, value='horny', label='Subscriber Archetype', scale=2)
            new_session_btn = gr.Button('New Session', scale=1)
            save_btn = gr.Button('Save Session', variant='primary', scale=1)
        chatbot = gr.Chatbot(value=[{'role': 'assistant', 'content': ARCHETYPES['horny']['opener']}], label='Chat', height=500)
        msg_input = gr.Textbox(placeholder='Type as Jasmin...', label='Your message (as Jasmin)', show_label=False)
        status_output = gr.Textbox(label='Status', interactive=False)
        archetype_dropdown.change(fn=on_archetype_change, inputs=[archetype_dropdown], outputs=[chatbot])
        msg_input.submit(fn=user_sends_message, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
        save_btn.click(fn=on_save_click, inputs=[chatbot], outputs=[status_output])
        new_session_btn.click(fn=on_new_session, inputs=[archetype_dropdown], outputs=[chatbot, status_output])

    try:
        demo.launch(share=IS_COLAB)
    except Exception:
        demo.launch(share=False)
else:
    print('Skipped subscriber sim UI — TRAINING_MODE is on.')

In [ ]:
# ── Cell 6: LoRA Adapter Setup ─────────────────────────────────────
# Applies Low-Rank Adaptation to the base model for efficient fine-tuning.
# Only ~1-2% of parameters become trainable.

if IS_COLAB and TRAINING_MODE:
    model = FastLanguageModel.get_peft_model(
        model,
        r              = 16,
        target_modules = [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj',
        ],
        lora_alpha     = 16,
        lora_dropout   = 0,
        bias           = 'none',
        use_gradient_checkpointing = 'unsloth',
        random_state   = 3407,
    )
    model.print_trainable_parameters()
else:
    print('Skipped — set TRAINING_MODE=True and run on Colab.')

In [ ]:
# ── Cell 7: Load & Format Training Data ────────────────────────────
# Reads per-archetype JSONL files and converts each session into a single
# formatted chat-template string for SFTTrainer.

from datasets import Dataset

def load_training_data():
    """Load all archetype JSONL files and format as HuggingFace Dataset."""

    # Check which archetype files exist
    available = []
    for name in ARCHETYPE_NAMES:
        f = archetype_file(name)
        if f.exists():
            available.append((name, f))

    if not available:
        if IS_COLAB:
            from google.colab import files
            print('No archetype JSONL files found on Drive.')
            print('Upload them now (e.g. horny.jsonl, simp.jsonl):')
            uploaded = files.upload()
            for fname, data in uploaded.items():
                dest = DATA_DIR / fname
                dest.write_bytes(data)
                print(f'Saved {fname} → {dest}')
            # Re-check
            for name in ARCHETYPE_NAMES:
                f = archetype_file(name)
                if f.exists():
                    available.append((name, f))
        else:
            raise FileNotFoundError(
                'No archetype files found. Run scripts/parse_chats.py first.'
            )

    # Load all sessions from all archetype files
    sessions = []
    for name, filepath in available:
        count = 0
        with open(filepath) as f:
            for line in f:
                sessions.append(json.loads(line))
                count += 1
        print(f'  {name:12s}: {count:4d} sessions from {filepath.name}')

    print(f'Loaded {len(sessions)} total sessions across {len(available)} archetypes')

    # Convert each session's messages to a formatted chat template string
    formatted = []
    skipped = 0
    for sess in sessions:
        msgs = sess['messages']
        # Ensure valid structure: system + at least user + assistant
        if len(msgs) < 3:
            skipped += 1
            continue
        if msgs[0]['role'] != 'system':
            skipped += 1
            continue

        # Strip any extra keys (timestamp etc.) — keep only role + content
        clean_msgs = [{'role': m['role'], 'content': m['content']} for m in msgs]

        text = tokenizer.apply_chat_template(
            clean_msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted.append({'text': text})

    print(f'Formatted {len(formatted)} sessions ({skipped} skipped)')

    # Token length stats
    lengths = []
    for item in formatted:
        toks = tokenizer(item['text'], truncation=False)['input_ids']
        lengths.append(len(toks))
    avg_len = sum(lengths) / len(lengths) if lengths else 0
    over_max = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
    print(f'Token lengths — avg: {avg_len:.0f}, max: {max(lengths)}, '
          f'over {MAX_SEQ_LENGTH}: {over_max}')

    return Dataset.from_list(formatted)


if IS_COLAB and TRAINING_MODE:
    train_dataset = load_training_data()
    print(f'\nDataset ready: {len(train_dataset)} examples')
    print(f'Sample (first 300 chars):\n{train_dataset[0]["text"][:300]}')
else:
    train_dataset = None
    print('Skipped — set TRAINING_MODE=True and run on Colab.')

In [ ]:
# ── Cell 8: Fine-Tune with SFTTrainer ──────────────────────────────
# Trains the LoRA adapter on subscriber chat data.
# ~15-30 min on Colab T4, ~5-10 min on A100.

if IS_COLAB and TRAINING_MODE:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import is_bfloat16_supported

    # Adapter save path (persisted to Google Drive)
    ADAPTER_DIR = DATA_DIR.parent / 'models' / 'subscriber-lora'
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

    trainer = SFTTrainer(
        model           = model,
        tokenizer       = tokenizer,
        train_dataset   = train_dataset,
        dataset_text_field = 'text',
        max_seq_length  = MAX_SEQ_LENGTH,
        dataset_num_proc = 2,
        packing         = False,
        args = TrainingArguments(
            per_device_train_batch_size = 1,   # reduced from 2 for T4 VRAM
            gradient_accumulation_steps = 8,   # doubled to keep effective batch=8
            warmup_steps     = 5,
            num_train_epochs = 3,
            learning_rate    = 2e-4,
            fp16             = not is_bfloat16_supported(),
            bf16             = is_bfloat16_supported(),
            logging_steps    = 1,
            optim            = 'adamw_8bit',
            weight_decay     = 0.01,
            lr_scheduler_type = 'linear',
            seed             = 3407,
            output_dir       = 'outputs',
            report_to        = 'none',
        ),
    )

    print('Starting training...')
    stats = trainer.train()
    print(f'\nTraining complete!')
    print(f'  Total steps:  {stats.global_step}')
    print(f'  Final loss:   {stats.training_loss:.4f}')
    print(f'  Runtime:      {stats.metrics["train_runtime"]:.0f}s')

    # Save adapter to Google Drive
    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    print(f'\nAdapter saved to: {ADAPTER_DIR}')
else:
    print('Skipped — set TRAINING_MODE=True and run on Colab.')

In [ ]:
# ── Cell 9: Test Fine-Tuned Model ──────────────────────────────────
# Quick inference test — generates subscriber responses to sample Jasmin messages.
# Tests each archetype to verify the model learned distinct personalities.

TEST_JASMIN_MSGS = [
    'hey babe welcome to my page 😘',
    'wanna see something special? i got a new vid just for you 🔥',
    'that'll be $25 for the full set baby',
    'you're so sweet, thanks for subscribing ❤️',
]

if IS_COLAB and model is not None:
    FastLanguageModel.for_inference(model)
    print('Switched to inference mode.\n')

    # Test 3 archetypes to show personality differences
    for arch_key in ['horny', 'cheapskate', 'cold']:
        arch = ARCHETYPES[arch_key]
        print(f'━━━ {arch["label"]} ━━━')

        for jasmin_msg in TEST_JASMIN_MSGS:
            messages = [
                {'role': 'system', 'content': arch['system']},
                {'role': 'user', 'content': jasmin_msg},
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors='pt',
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    input_ids=inputs,
                    max_new_tokens=150,
                    temperature=0.85,
                    top_p=0.9,
                    do_sample=True,
                    repetition_penalty=1.1,
                )

            response = tokenizer.decode(
                outputs[0][inputs.shape[-1]:],
                skip_special_tokens=True,
            ).strip()

            print(f'  JAS: {jasmin_msg}')
            print(f'  SUB: {response}')
            print()
else:
    print('Skipped — run on Colab after training.')

In [ ]:
# ── Cell 10: Subscriber Sim (Post-Training) ───────────────────────
# Same subscriber sim as Cell 5 — you type as Jasmin, model plays subscriber.
# Runs after training so you can keep collecting data with the model in
# inference mode. Data you collect here further trains Jasmin.

import gradio as gr

current_archetype = 'horny'
current_session_id = str(uuid.uuid4())[:8]


def on_archetype_change(archetype_key):
    global current_archetype, current_session_id
    current_archetype = archetype_key
    current_session_id = str(uuid.uuid4())[:8]
    return [{'role': 'assistant', 'content': ARCHETYPES[archetype_key]['opener']}]


def user_sends_message(user_message, history):
    if not history:
        history = []
    history.append({'role': 'user', 'content': user_message})
    subscriber_reply = generate_subscriber_message(
        assistant_message=user_message,
        history=history,
        archetype_key=current_archetype,
    )
    history.append({'role': 'assistant', 'content': subscriber_reply})
    return '', history


def on_save_click(history):
    return save_session(history, current_archetype, current_session_id)


def on_new_session(archetype_key):
    global current_session_id
    current_session_id = str(uuid.uuid4())[:8]
    return [{'role': 'assistant', 'content': ARCHETYPES[archetype_key]['opener']}], ''


archetype_choices = [(v['label'], k) for k, v in ARCHETYPES.items()]

with gr.Blocks(title='Subscriber Simulator') as demo:
    gr.Markdown('# Subscriber Simulator\nYou are **Jasmin**. The bot is a subscriber. Select an archetype and chat.')
    with gr.Row():
        archetype_dropdown = gr.Dropdown(choices=archetype_choices, value='horny', label='Subscriber Archetype', scale=2)
        new_session_btn = gr.Button('New Session', scale=1)
        save_btn = gr.Button('Save Session', variant='primary', scale=1)
    chatbot = gr.Chatbot(value=[{'role': 'assistant', 'content': ARCHETYPES['horny']['opener']}], label='Chat', height=500)
    msg_input = gr.Textbox(placeholder='Type as Jasmin...', label='Your message (as Jasmin)', show_label=False)
    status_output = gr.Textbox(label='Status', interactive=False)
    archetype_dropdown.change(fn=on_archetype_change, inputs=[archetype_dropdown], outputs=[chatbot])
    msg_input.submit(fn=user_sends_message, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
    save_btn.click(fn=on_save_click, inputs=[chatbot], outputs=[status_output])
    new_session_btn.click(fn=on_new_session, inputs=[archetype_dropdown], outputs=[chatbot, status_output])

try:
    demo.launch(share=IS_COLAB)
except Exception:
    demo.launch(share=False)